In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from tqdm import tqdm

# ==========================================
# 1. 환경 및 경로 설정
# ==========================================
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

DATA_DIR = r"/home/a202192020/맥주데이터실험/data"
OUTPUT_DIR = r"/home/a202192020/맥주데이터실험/ver3_gemini/output"
XLSX_NAME = "Supplemental Files and Figure source files.xlsx"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. 데이터 로드 (X와 Y를 모두 포함)
# ==========================================
def load_data_full(data_dir, filename):
    path = os.path.join(data_dir, filename)
    if not os.path.exists(path): raise FileNotFoundError(f"❌ 파일 없음: {path}")
    
    print(f"📂 데이터 로딩 중... ({filename})")
    xls = pd.ExcelFile(path)
    df_chem = xls.parse("Supplementary File S1") # X
    df_panel = xls.parse("Supplementary File S4") # Y
    
    if 'beer_id' in df_chem.columns: df_chem = df_chem.set_index('beer_id')
    if 'beer_id' in df_panel.columns: df_panel = df_panel.set_index('beer_id')
    
    # 교집합 병합
    df_merged = df_chem.join(df_panel, how='inner', lsuffix='_chem', rsuffix='_panel')
    
    # 스타일 컬럼 찾기
    style_col = 'tasting_category_fine'
    if style_col not in df_merged.columns:
        candidates = [c for c in df_merged.columns if 'category' in c or 'style' in c]
        style_col = candidates[0] if candidates else None
        
    return df_merged, style_col

# 데이터 로드
df_all, style_col = load_data_full(DATA_DIR, XLSX_NAME)

# 타겟 스타일 선정
target_style = df_all[style_col].value_counts().idxmax()
print(f"🎯 **타겟 스타일:** [{target_style}] (이 데이터만 학습합니다)")

# 타겟 스타일만 필터링
df_target = df_all[df_all[style_col] == target_style].copy()

# -------------------------------------------------------
# [핵심 변경] X(화학)와 Y(맛)를 모두 합쳐서 학습 데이터로 만듦
# -------------------------------------------------------
# X 컬럼 정의
try:
    x_start = df_target.columns.get_loc("acetaldehyde")
    x_end = df_target.columns.get_loc("sulfur_sum")
    X_cols = df_target.columns[x_start : x_end+1]
except KeyError:
    numeric_cols = df_target.select_dtypes(include=[np.number]).columns
    X_cols = numeric_cols[:231]

# Y 컬럼 정의
Y_cols = [c for c in df_target.columns if c not in X_cols and c != style_col and df_target[c].dtype != 'object']

# 데이터셋 구성 (X + Y)
data_target = df_target[list(X_cols) + Y_cols].fillna(0) # 결측치 0 처리
print(f"📊 학습 데이터 크기 (X+Y): {data_target.shape}")
print(f"   - X(화학) 컬럼 수: {len(X_cols)}")
print(f"   - Y(맛)   컬럼 수: {len(Y_cols)}")

# 스케일링 (전체 0~1)
# (이미 로그변환된 X와 일반 점수 Y를 모두 0~1로 맞춤)
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data_target)

# Train/Test 분할 (생성 모델 학습용 Train)
train_data, test_data = train_test_split(data_scaled, test_size=0.2, random_state=42)

# Tensor 변환
train_tensor = torch.FloatTensor(train_data).to(device)
train_loader = DataLoader(TensorDataset(train_tensor), batch_size=32, shuffle=True)

# ==========================================
# 3. 모델 정의 (VAE + Diffusion) -> X,Y 동시 학습
# ==========================================
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super(VAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU()
        )
        self.fc_mu = nn.Linear(64, latent_dim)
        self.fc_var = nn.Linear(64, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, input_dim), nn.Sigmoid()
        )
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_var(h)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    def decode(self, z):
        return self.decoder(z)
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

class LatentDiffusion(nn.Module):
    def __init__(self, latent_dim=16, time_steps=1000):
        super(LatentDiffusion, self).__init__()
        self.time_steps = time_steps
        self.time_embed = nn.Sequential(
            nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 16)
        )
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 16, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
    def forward(self, x, t):
        t_embed = self.time_embed(t.float() / self.time_steps)
        return self.net(torch.cat([x, t_embed], dim=1))

# ==========================================
# 4. 학습 (Training)
# ==========================================
latent_dim = 16
input_dim = train_data.shape[1] # X+Y 전체 차원

vae = VAE(input_dim, latent_dim).to(device)
optimizer_vae = optim.Adam(vae.parameters(), lr=1e-3)

print("\n🚀 [Step 1] VAE 학습 (X, Y 동시 압축)")
epochs_vae = 2000
for epoch in tqdm(range(epochs_vae), desc="VAE Training"):
    vae.train()
    total_loss = 0
    for batch in train_loader:
        x = batch[0]
        optimizer_vae.zero_grad()
        recon_x, mu, logvar = vae(x)
        recon_loss = nn.functional.mse_loss(recon_x, x, reduction='sum')
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        loss = recon_loss + 0.001 * kl_loss
        loss.backward()
        optimizer_vae.step()

# Latent 추출
vae.eval()
with torch.no_grad():
    latents = vae.encode(train_tensor)[0]

diffusion = LatentDiffusion(latent_dim).to(device)
optimizer_diff = optim.Adam(diffusion.parameters(), lr=1e-3)
beta = torch.linspace(1e-4, 0.02, 1000).to(device)
alpha = 1 - beta
alpha_hat = torch.cumprod(alpha, dim=0)

print("\n🚀 [Step 2] Diffusion 학습 (Latent 분포 학습)")
epochs_diff = 3000
latent_loader = DataLoader(TensorDataset(latents), batch_size=32, shuffle=True)

for epoch in tqdm(range(epochs_diff), desc="Diffusion Training"):
    diffusion.train()
    for batch in latent_loader:
        z0 = batch[0]
        t = torch.randint(0, 1000, (z0.size(0), 1)).to(device)
        noise = torch.randn_like(z0).to(device)
        a_bar = alpha_hat[t]
        zt = torch.sqrt(a_bar) * z0 + torch.sqrt(1 - a_bar) * noise
        
        optimizer_diff.zero_grad()
        noise_pred = diffusion(zt, t)
        loss = nn.functional.mse_loss(noise_pred, noise)
        loss.backward()
        optimizer_diff.step()

# ==========================================
# 5. 생성 및 평가 (Evaluation)
# ==========================================
print("\n✨ [Step 3] 데이터 생성 (Generation)")
num_samples = 200
diffusion.eval()
vae.eval()

with torch.no_grad():
    z = torch.randn(num_samples, latent_dim).to(device)
    for i in reversed(range(1000)):
        t = torch.full((num_samples, 1), i).to(device)
        predicted_noise = diffusion(z, t)
        beta_t = beta[i]
        alpha_t = alpha[i]
        alpha_hat_t = alpha_hat[i]
        if i > 0: noise = torch.randn_like(z)
        else: noise = 0
        z = (1 / torch.sqrt(alpha_t)) * (z - (beta_t / torch.sqrt(1 - alpha_hat_t)) * predicted_noise) + torch.sqrt(beta_t) * noise
        
    generated_data_scaled = vae.decode(z).cpu().numpy()
    generated_data = scaler.inverse_transform(generated_data_scaled)
    
    # DataFrame 변환 (컬럼명 복구)
    df_gen = pd.DataFrame(generated_data, columns=list(X_cols) + Y_cols)

# 저장
save_path = os.path.join(OUTPUT_DIR, f"generated_{target_style}_XY_Direct.csv")
df_gen.to_csv(save_path, index=False)
print(f"💾 생성된 데이터 저장 완료: {save_path}")

# ==========================================
# 6. 최종 성능 평가 (XGBoost)
# ==========================================
print("\n⚖️ [Final Eval] XGBoost로 성능 평가 (Real vs Real+Syn)")

# 생성된 데이터를 X와 Y로 다시 분리
X_syn = df_gen[X_cols]
Y_syn = df_gen[Y_cols]

# 원본 타겟 스타일 데이터 (Test용)
X_real_target = df_target[X_cols].fillna(0)
Y_real_target = df_target[Y_cols].fillna(0)
X_train_r, X_test_r, Y_train_r, Y_test_r = train_test_split(X_real_target, Y_real_target, test_size=0.2, random_state=42)

# 평가 모델 (XGBoost)
evaluator = MultiOutputRegressor(XGBRegressor(n_estimators=100, n_jobs=-1, random_state=42))

# 1) Baseline: 원본만 학습
evaluator.fit(X_train_r, Y_train_r)
r2_base = r2_score(Y_test_r, evaluator.predict(X_test_r))

# 2) Augmented: 원본 + 생성 데이터 학습
X_train_aug = pd.concat([X_train_r, X_syn])
Y_train_aug = pd.concat([Y_train_r, Y_syn])

evaluator.fit(X_train_aug, Y_train_aug)
r2_aug = r2_score(Y_test_r, evaluator.predict(X_test_r))

print(f"\n📊 결과 리포트 ({target_style})")
print(f"   - Baseline (Real Only) R2: {r2_base:.4f}")
print(f"   - Augmented (Real+Syn) R2: {r2_aug:.4f}")
print(f"   🚀 성능 향상 (Gain)      : {r2_aug - r2_base:.4f}")

📂 데이터 로딩 중... (Supplemental Files and Figure source files.xlsx)
🎯 **타겟 스타일:** [Blond] (이 데이터만 학습합니다)
📊 학습 데이터 크기 (X+Y): (31, 283)
   - X(화학) 컬럼 수: 231
   - Y(맛)   컬럼 수: 52

🚀 [Step 1] VAE 학습 (X, Y 동시 압축)


VAE Training: 100%|██████████| 2000/2000 [00:17<00:00, 115.42it/s]



🚀 [Step 2] Diffusion 학습 (Latent 분포 학습)


Diffusion Training: 100%|██████████| 3000/3000 [00:15<00:00, 191.74it/s]



✨ [Step 3] 데이터 생성 (Generation)
💾 생성된 데이터 저장 완료: /home/a202192020/맥주데이터실험/ver3_gemini/output/generated_Blond_XY_Direct.csv

⚖️ [Final Eval] XGBoost로 성능 평가 (Real vs Real+Syn)

📊 결과 리포트 (Blond)
   - Baseline (Real Only) R2: -38.7658
   - Augmented (Real+Syn) R2: -38.7457
   🚀 성능 향상 (Gain)      : 0.0201
